In [4]:
import pandas as pd
import re

billets_df = pd.read_csv('billets.csv', sep=',')

# Create initial ID
billets_df['ID'] = billets_df['CODE'].astype(str) + '-' + billets_df['YEAR'].astype(str) + '-' + billets_df['CITY'].astype(str)

# If not unique, add -2, -3, etc. to duplicates
billets_df['ID'] = billets_df.groupby('ID').cumcount().replace(0, '', regex=True).astype(str).radd(billets_df['ID']).replace(r'(\D)$', r'\1', regex=True)
billets_df['ID'] = billets_df['ID'].str.replace(r'(\D+)$', '', regex=True)  # Remove trailing empty string

# Fix IDs: add -2, -3, etc. only to duplicates
def fix_id(row):
    base_id = f"{row['CODE']}-{row['YEAR']}-{row['CITY']}"
    count = billets_df.loc[
        (billets_df['CODE'] == row['CODE']) &
        (billets_df['YEAR'] == row['YEAR']) &
        (billets_df['CITY'] == row['CITY']) &
        (billets_df.index <= row.name)
    ].shape[0]
    return base_id if count == 1 else f"{base_id}-{count}"

billets_df['ID'] = billets_df.apply(fix_id, axis=1)

is_id_unique = billets_df['ID'].is_unique
print(f"Is 'ID' column unique? {is_id_unique}")

# Show duplicate base IDs (before suffixes)
base_ids = billets_df['CODE'].astype(str) + '-' + billets_df['YEAR'].astype(str) + '-' + billets_df['CITY'].astype(str)
duplicate_base_ids = billets_df[base_ids.duplicated(keep=False)]
print("Duplicate base IDs:")
print(duplicate_base_ids[['ID', 'CODE', 'YEAR', 'CITY']])


def extract_source_id(info_link):
    match = re.search(r'billet-(\d+)', info_link)
    return match.group(1) if match else None

billets_df['SOURCE_ID'] = billets_df['INFO_LINK'].apply(extract_source_id)

billets_df['IMAGE_RECTO'] = billets_df['SOURCE_ID'] + '_recto.jpg'
billets_df['IMAGE_VERSO'] = billets_df['SOURCE_ID'] + '_verso.jpg'
billets_df = billets_df.drop(columns=['INFO_LINK'])
billets_df

Is 'ID' column unique? True
Duplicate base IDs:
                               ID  CODE    YEAR           CITY
1228    XEWD-2023-1-OELSNITZVOGTL  XEWD  2023-1  OELSNITZVOGTL
1229  XEWD-2023-1-OELSNITZVOGTL-2  XEWD  2023-1  OELSNITZVOGTL


,POSTAL_CODE,CODE,YEAR,CITY,TITLE,ID,SOURCE_ID,IMAGE_RECTO,IMAGE_VERSO
0,IS-1,IS--,2022-1,REYKJAVIK,PF 2022,IS---2022-1-REYKJAVIK,2740,2740_recto.jpg,2740_verso.jpg
1,IS-1,ISAA,2022-1,REYKJAVIK,REYKJAVIK,ISAA-2022-1-REYKJAVIK,2739,2739_recto.jpg,2739_verso.jpg
2,MG-A,MGAA,2021-1,MAROANTSETRA,THE KING OF MADAGASCARMORIC BENOVSKY 275th BIR...,MGAA-2021-1-MAROANTSETRA,2465,2465_recto.jpg,2465_verso.jpg
3,IQ-00,IQAA,2019-1,HILLA,IRAQ - ISHTAR GATE OF BABYLONWORLD HERITAGE,IQAA-2019-1-HILLA,1502,1502_recto.jpg,1502_verso.jpg
4,IQ-00,IQAB,2019-1,HILLA,IRAQ - HANGING GARDENS OF BABYLONWORLD HERITAGE,IQAB-2019-1-HILLA,1503,1503_recto.jpg,1503_verso.jpg
...,...,...,...,...,...,...,...,...,...
3291,VA,SEFJ,2024-1,VATICAN,VATICANOIN RICORDO DEL GIUBILEO 2025,SEFJ-2024-1-VATICAN,4418,4418_recto.jpg,4418_verso.jpg
3292,VA,SEGY,2018-2,VATICANO,VATICANO,SEGY-2018-2-VATICANO,637,637_recto.jpg,637_verso.jpg
3293,LU-VD,REAC,2022-1,VIANDEN,BURG VIANDEN - CHÂTEAU VIANDEN,REAC-2022-1-VIANDEN,3775,3775_recto.jpg,3775_verso.jpg
3294,ZA-WC,JEAA,2022-1,GANSBAAI,GREAT WHITE SHARKGANSBAAI,JEAA-2022-1-GANSBAAI,2775,2775_recto.jpg,2775_verso.jpg


In [3]:
billets_json = billets_df.to_json(orient='records', force_ascii=False)
with open('billets.json', 'w', encoding='utf-8') as f:
    f.write(billets_json)
billets_json

'[{"POSTAL_CODE":"IS-1","CODE":"IS--","YEAR":"2022-1","CITY":"REYKJAVIK","TITLE":"PF 2022","INFO_LINK":"https:\\/\\/www.billets-touristiques.com\\/billet-2740","ID":"IS---2022-1-REYKJAVIK","SOURCE_ID":"2740","IMAGE_RECTO":"2740_recto.jpg","IMAGE_VERSO":"2740_verso.jpg"},{"POSTAL_CODE":"IS-1","CODE":"ISAA","YEAR":"2022-1","CITY":"REYKJAVIK","TITLE":"REYKJAVIK","INFO_LINK":"https:\\/\\/www.billets-touristiques.com\\/billet-2739","ID":"ISAA-2022-1-REYKJAVIK","SOURCE_ID":"2739","IMAGE_RECTO":"2739_recto.jpg","IMAGE_VERSO":"2739_verso.jpg"},{"POSTAL_CODE":"MG-A","CODE":"MGAA","YEAR":"2021-1","CITY":"MAROANTSETRA","TITLE":"THE KING OF MADAGASCARMORIC BENOVSKY 275th BIRTH ANNIVERSARY","INFO_LINK":"https:\\/\\/www.billets-touristiques.com\\/billet-2465","ID":"MGAA-2021-1-MAROANTSETRA","SOURCE_ID":"2465","IMAGE_RECTO":"2465_recto.jpg","IMAGE_VERSO":"2465_verso.jpg"},{"POSTAL_CODE":"IQ-00","CODE":"IQAA","YEAR":"2019-1","CITY":"HILLA","TITLE":"IRAQ - ISHTAR GATE OF BABYLONWORLD HERITAGE","INFO_LI